# Step 3 -- Ancestral GOEA with EAGLE (Extant and Ancestral Gene List Enrichment)

In [39]:
import os
import subprocess

Path definitions:

In [40]:
common_data_path = '/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData'

nwk_fn = os.path.join(common_data_path, 'Module2', 'species_tree.nwk')  # might need to be the species_tree_checked.nwk from FastOMA output
oxml_fn = os.path.join(common_data_path, 'Module2', 'fastoma_output', 'FastOMA_HOGs.orthoxml')

obo_fn = os.path.join(common_data_path, 'Module4', 'go-basic.obo')
hogprop_results_fn = os.path.join(common_data_path, 'Module4', 'precomputed_results', 'hogprop_output.h5')
#'../step2_propagation/hogprop_output.h5'

In [41]:
hogprop_results_fn

'/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData/Module4/precomputed_results/hogprop_output.h5'

In [42]:
!cat {nwk_fn}

((Crocodylus_porosus:1,(((((((Aptenodytes_forsteri :1, Oreotrochilus_melanogaster :1)Neoaves_4c:1, Grus_americana :1)Neoaves_3c:1, Taeniopygia_guttata :1)Neoaves_2a:1, Columba_livia :1)Neoaves_1a:1, Phoenicopterus_ruber :1)Neoaves:1, Gallus_gallus :1)Neognathae:1, Struthio_camelus:1)Aves:0)Archosauria:1, Homo_sapiens:1);


We need the newick string from FastOMA results (species_tree_checked.nwk) as this has the same internal node names as the OrthoXML. Below we create a variable called nwk where we copy and paste the tree string.

In [43]:
nwk = '((Crocodylus_porosus:1,(((((((Aptenodytes_forsteri :1, Oreotrochilus_melanogaster :1)Neoaves_4c:1, Grus_americana :1)Neoaves_3c:1, Taeniopygia_guttata :1)Neoaves_2a:1, Columba_livia :1)Neoaves_1a:1, Phoenicopterus_ruber :1)Neoaves:1, Gallus_gallus :1)Neognathae:1, Struthio_camelus:1)Aves:0)Archosauria:1, Homo_sapiens:1)internal_0;'

In [44]:
#write species tree newick to current path

with open('species_tree.nwk', 'wt') as fp:
    print(nwk, file=fp)

# Method to run enrichment on single branch

## Running GO enrichment on a single branch

We will begin by performing GO enrichment analysis on a single branch of the bird phylogeny.

In this example, we compare the ancestral node **Archoasauria** (ancestral to all birds and crocodiles) to **Aves**, a large clade containing the modern bird species. By examining the HOGs inferred to be gained, duplicated, lost, or retained along this branch, we can identify biological functions that may have been important during the diversification of Neoaves.

EAGLE performs enrichment analysis using:

- the species tree,
- the HOG structure inferred by FastOMA,
- functional annotations propagated with HogProp,
- and the Gene Ontology hierarchy.

The `--parent_name` and `--child_name` parameters define the branch of interest. EAGLE will automatically determine which evolutionary events occurred between these two nodes and test whether particular GO terms are overrepresented among the affected HOGs.

For larger analyses, EAGLE can also be run on all branches of a phylogeny, but starting with a single branch makes it easier to understand the workflow and inspect the results.

In [45]:
%%bash
which python
which jupyter-lab
which eagle
which hogprop

/home/ubuntu/miniconda3/envs/module4env/bin/python
/home/ubuntu/miniconda3/envs/module4env/bin/eagle
/home/ubuntu/miniconda3/envs/module4env/bin/hogprop


In [46]:
import os

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [48]:
os.makedirs("eagle_branch_path_results", exist_ok=True)

OBO = obo_fn
ORTHOXML = oxml_fn
TREE = "species_tree.nwk"
HOGPROP = hogprop_results_fn
print(HOGPROP)
OUTDIR = "eagle_branch_path_results"

cmd = [
    "eagle",
    "--obo", OBO,
    "--oxml", ORTHOXML,
    "--nwk", TREE,
    "--hogprop_results", HOGPROP,
    "--results", OUTDIR,
    "--include_genelist",
    "--skip_terminal",
    "--write_extant_genelist",
    "--parent_name", "Archosauria",
    "--child_name", "Aves"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData/Module4/precomputed_results/hogprop_output.h5
Loaded data in 14.069941 seconds
DONE! Took 598.880492s.



---

Alternatively, submit the script `sbatch eagle.sh` in order to run on every evolutionary event set of genes in the tree (internal branches only).

---

Now, to read the results we can analyse individual files / branches, e.g., 

In [52]:
from eagle.results import ResultsReader

In [53]:
!ls eagle_branch_path_results

Archosauria_Aves_duplicated.extant_genelist.tsv.gz
Archosauria_Aves_duplicated.tsv.gz
Archosauria_Aves_gained.extant_genelist.tsv.gz
Archosauria_Aves_gained.tsv.gz
Archosauria_Aves_lost.extant_genelist.tsv.gz
Archosauria_Aves_lost.tsv.gz
Archosauria_Aves_retained.extant_genelist.tsv.gz
Archosauria_Aves_retained.tsv.gz


In [54]:
r = ResultsReader('eagle_branch_path_results/Archosauria_Aves_duplicated.tsv.gz')

In [ ]:
r.header

{'eagle_version': '0.0.2',
 'time_run': '2026-06-16 09:53:29.582579',
 'time_taken': '140.158488',
 'branch_type': 'None',
 'event_type': 'duplicated',
 'parent_name': 'Archosauria',
 'child_name': 'Aves'}

In [ ]:
df = r.read()

In [ ]:
df.head()

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,study_proportion,population_count,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term
0,GO:0010482,regulation of epidermal cell division,BP,5,1.801313e-10,4.179046e-08,6.138752e-09,232,232,232 / 232,1.0,15065,16583,15065 / 16583,0.908460,1.100763,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
1,GO:0010839,negative regulation of keratinocyte proliferation,BP,7,1.916660e-10,4.446652e-08,6.138752e-09,232,232,232 / 232,1.0,15069,16583,15069 / 16583,0.908702,1.100471,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
2,GO:0061436,establishment of skin barrier,BP,6,1.916660e-10,4.446652e-08,6.138752e-09,232,232,232 / 232,1.0,15069,16583,15069 / 16583,0.908702,1.100471,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
3,GO:0010837,regulation of keratinocyte proliferation,BP,6,1.946628e-10,4.516178e-08,6.138752e-09,232,232,232 / 232,1.0,15070,16583,15070 / 16583,0.908762,1.100398,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
4,GO:0045606,positive regulation of epidermal cell differen...,BP,6,1.946628e-10,4.516178e-08,6.138752e-09,232,232,232 / 232,1.0,15070,16583,15070 / 16583,0.908762,1.100398,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...


Or we can read many at once:

In [55]:
from eagle.results import parse_to_concat
import gzip

In [ ]:
goea_path = 'eagle_branch_path_results'
fns = list(map(lambda x: os.path.join(goea_path, x),
               filter(lambda x: not x.endswith('.extant_genelist.tsv.gz'),
                      os.listdir(goea_path))))
df = parse_to_concat(*fns)

with gzip.open('eagle_results_with_ancestral_genes.tsv.gz', 'wt') as fp:
    df.to_csv(fp, sep='\t', index=False)

with gzip.open('eagle_results.tsv.gz', 'wt') as fp:
    header = [x for x in list(df) if x != 'study_entries_with_go_term']
    df[header].to_csv(fp, sep='\t', index=False)

100%|██████████| 4/4 [00:00<00:00,  4.40it/s]


In the previous step, we identified GO terms that are significantly enriched among HOGs that were gained, duplicated, retained, or lost along the branch leading from Aves to Neoaves. However, enrichment analyses often produce long lists of highly redundant GO terms because many terms describe related biological processes.

To simplify interpretation, we will first reduce GO term redundancy using GO-Figure! and then explore the major functional themes associated with each event type.

## Step 1. Filter significant enrichments

For each event type, retain only significantly enriched GO terms:

In [56]:
sig_df = df[df['p_bonferroni']<=0.05]

How many significantly enriched GO terms are associated with gained, duplicated, retained, and lost HOGs?

## Step 2. Reduce redundancy with GO-Figure!

Use GO-Figure! to cluster related GO terms and generate summary plots.

For each event type:

- gained
- duplicated
- retained
- lost

generate:

A GO-Figure! input file
A GO-Figure! summary plot

In [58]:
sig_df.columns

Index(['GO_ID', 'GO_name', 'GO_aspect', 'GO_depth', 'p_uncorrected',
       'p_bonferroni', 'p_fdr_bh', 'study_count', 'study_size', 'study_ratio',
       'study_proportion', 'population_count', 'population_size',
       'population_ratio', 'population_proportion', 'fold_change',
       'study_entries_with_go_term', 'branch_type', 'event_type',
       'parent_name', 'child_name'],
      dtype='object')

In [59]:
import os

os.makedirs("gofigure_inputs", exist_ok=True)
os.makedirs("gofigure_outputs", exist_ok=True)

#needed for go-figure
- python -m pip install scikit-learn
- python -m pip install seaborn

In [60]:
event_types = ['gained', 'lost', 'retained', 'duplicated']

for event_type in event_types:
    tmp_df = sig_df[
        (sig_df["event_type"] == event_type) &
        (sig_df['parent_name']=='Archosauria') &
        (sig_df['child_name']=='Aves')
    ]
    
    gofigure_input = tmp_df[["GO_ID", "p_bonferroni"]]
    
    gofigure_input.to_csv(
        "./gofigure_inputs/{}.tsv".format(event_type),
        sep="\t",
        index=False,
        header=False
    )

In [67]:
import subprocess

for event_type in event_types:

    cmd = [
        "python",
        "../gits/GO-Figure/gofigure.py",
        "-i", "gofigure_inputs/{}.tsv".format(event_type),
        "-o", "gofigure_outputs/gofigure_{}.tsv".format(event_type)
    ]
    print(cmd)
    subprocess.run(cmd, check=True)

['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/gained.tsv', '-o', 'gofigure_outputs/gofigure_gained.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_gained.tsv/biological_process_gofigure_outputs_gofigure_gained.tsv.png

Calculating molecular function
Output for molecular functionfound in: gofigure_outputs/gofigure_gained.tsv/molecular_function_gofigure_outputs_gofigure_gained.tsv.png

Calculating cellular component
Output for cellular componentfound in: gofigure_outputs/gofigure_gained.tsv/cellular_component_gofigure_outputs_gofigure_gained.tsv.png
Finished!


['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/lost.tsv', '-o', 'gofigure_outputs/gofigure_lost.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_lost.tsv/biological_process_gofigure_outputs_gofigure_lost.tsv.png

Calculating molecular function
Output for molecular functionfound in: gofigure_outputs/gofigure_lost.tsv/molecular_function_gofigure_outputs_gofigure_lost.tsv.png

Calculating cellular component
Output for cellular componentfound in: gofigure_outputs/gofigure_lost.tsv/cellular_component_gofigure_outputs_gofigure_lost.tsv.png
Finished!


['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/retained.tsv', '-o', 'gofigure_outputs/gofigure_retained.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_retained.tsv/biological_process_gofigure_outputs_gofigure_retained.tsv.png

Calculating molecular function
Output for molecular functionfound in: gofigure_outputs/gofigure_retained.tsv/molecular_function_gofigure_outputs_gofigure_retained.tsv.png

Calculating cellular component
Output for cellular componentfound in: gofigure_outputs/gofigure_retained.tsv/cellular_component_gofigure_outputs_gofigure_retained.tsv.png
Finished!


['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/duplicated.tsv', '-o', 'gofigure_outputs/gofigure_duplicated.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_duplicated.tsv/biological_process_gofigure_outputs_gofigure_duplicated.tsv.png

Calculating molecular function
Output for molecular functionfound in: gofigure_outputs/gofigure_duplicated.tsv/molecular_function_gofigure_outputs_gofigure_duplicated.tsv.png

Calculating cellular component
/home/ubuntu/miniconda3/envs/module4env/lib/python3.9/site-packages/sklearn/manifold/_mds.py:166: RuntimeWarning: invalid value encountered in scalar divide
  old_stress = stress / dis
/home/ubuntu/miniconda3/envs/module4env/lib/python3.9/site-packages/sklearn/manifold/_mds.py:162: RuntimeWarning: invalid value encountered in scalar divide
  if (old_stress - stress / dis) < eps:
/home/ubuntu/miniconda3/envs/module4env/lib/python3.9/site-packages/sklearn/manifold/_mds.py:166: RuntimeWarning: invalid value encountered in scalar divide
  old_stress = stress / dis
/home/ubuntu/miniconda3/env

## Step 3. Examine major biological themes


Inspect the GO-Figure! plots and identify the most prominent functional categories.

Questions

- Which biological processes or cellular components appear most strongly enriched among gained HOGs?
- Which themes are associated with duplicated HOGs?
- Which functions are predominantly lost?
- Are there any biological themes that seem particularly relevant to bird evolution?

In [ ]:

sig_df[sig_df['GO_name'].str.contains('intermediate filament cytoskeleton')]

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,...,population_count,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term,branch_type,event_type,parent_name,child_name
656,GO:0045111,intermediate filament cytoskeleton,CC,6,3.076302e-69,2.098345e-65,1.140405e-67,106,473,106 / 473,...,365,13475,365 / 13475,0.027087,8.273335,HOG:0002308.1au.14b.6b.2b_6@Neoaves_1a;HOG:000...,internal,lost,Neoaves_1a,Neoaves_2a
1076,GO:0045111,intermediate filament cytoskeleton,CC,6,1.202346e-61,8.376747e-58,3.932745e-60,110,412,110 / 412,...,542,13660,542 / 13660,0.039678,6.728944,HOG:0002221.1d_2&23356158053824@Aves;HOG:00023...,internal,lost,Aves,Neognathae
3046,GO:0045111,intermediate filament cytoskeleton,CC,6,6.723169e-43,4.376111e-39,5.402606e-41,102,631,102 / 631,...,458,13781,458 / 13781,0.033234,4.863916,HOG:0002308.1au.14b_4&23210491411232@Neognatha...,internal,lost,Neognathae,Neoaves
5270,GO:0045111,intermediate filament cytoskeleton,CC,6,2.902338e-30,1.929764e-26,2.717978e-28,59,313,59 / 313,...,413,13495,413 / 13495,0.030604,6.159288,HOG:0002308.1au.14b.6b_5&23114211293936@Neoave...,internal,lost,Neoaves,Neoaves_1a
8835,GO:0045111,intermediate filament cytoskeleton,CC,6,2.048834e-20,1.705039e-16,3.755593e-19,103,878,103 / 878,...,604,13624,604 / 13624,0.044334,2.646125,HOG:0000787_1&22946410080720@Archosauria;HOG:0...,internal,lost,Archosauria,Aves
11448,GO:0045111,intermediate filament cytoskeleton,CC,6,1.069788e-15,7.337676e-12,8.272465e-15,71,1288,71 / 1288,...,274,13667,274 / 13667,0.020048,2.749572,HOG:0000785_1&22748233146128@Neoaves_2a;HOG:00...,internal,lost,Neoaves_2a,Neoaves_3c
16781,GO:0045111,intermediate filament cytoskeleton,CC,6,6.367088e-11,4.902021e-07,1.795360e-09,36,651,36 / 651,...,203,12580,203 / 12580,0.016137,3.426937,HOG:0001045_1&22947374139376@Neoaves_3c;HOG:00...,internal,lost,Neoaves_3c,Neoaves_4c
20984,GO:0045111,intermediate filament cytoskeleton,CC,6,3.973676e-08,4.000299e-04,7.859134e-07,23,843,23 / 843,...,82,10788,82 / 10788,0.007601,3.589445,HOG:0000787_1@internal_0;HOG:0001045_1@interna...,internal,duplicated,internal_0,Archosauria
